# Creating cell type specific fragment file for ChromBPnet

#### Problem:
many fragments have tons of reads... Is the read number column used for chrombpnet? Should they be kept at their number or set at max 1-4?

In [1]:
here::i_am("rna_atac/create_fragment_files/fragment_files.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘rtracklayer’ was built under R version 4.2.3”


In [2]:
args = list()
args$metadata = file.path(io$basedir, 'results/rna_atac/clustering/metadata_celltype_annotated.txt.gz')


In [3]:
args$metadata 

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/rna_atac/clustering/metadata_celltype_annotated.txt.gz"

In [5]:
meta = fread(file.path(io$basedir, 'results/rna_atac/clustering/metadata_celltype_annotated.txt.gz'))

Warning message in writeBin(bfr, con = out, size = 1L):
“problem writing to connection”


ERROR: Error in fread(file.path(io$basedir, "results/rna_atac/clustering/metadata_celltype_annotated.txt.gz")): File is empty: /tmp/RtmpAYGuIF/file23b93a68b5e08e


In [ ]:
unique(meta$celltype_v1)

In [ ]:
tmp = meta[celltype_v1 == 'Primitive_Streak']

In [ ]:
unique(tmp$sample)

In [ ]:
fragments_combined = mclapply(unique(tmp$sample)[1:3], function(x){ # unique(tmp$sample)
    print(x)
    tmp2 = fread(sprintf('/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/original/%s/outs/atac_fragments.tsv.gz', x)) %>% 
        .[V4 %in% unique(tmp$barcode)] %>% 
        .[, V4 := 'ACTG'] %>% # This is just a small string to make the file size smaller, since single-cells information isn't used downstream
        setnames(c('Chr', 'Start', 'End', 'Barcode', 'Reads'))
    print('done')
    return(tmp2)
}, mc.cores=1) %>% rbindlist() %>%
    .[order(c('Chr', 'Start', 'End'))]

In [ ]:
fwrite(fragments_combined, '/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code/rna_atac/create_fragment_files/test.tsv.gz', header = FALSE)